In [1]:
import gc
from functools import partial
from pathlib import Path

from fastai.text import *
from fastai.callbacks import *
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

ModuleNotFoundError: No module named 'fastai.callbacks'

In [2]:
path = Path('../input/google-quest-challenge')

In [3]:
train = pd.read_csv(path/'train.csv')
test = pd.read_csv(path/'test.csv')

NameError: name 'pd' is not defined

In [4]:
cols = ['question_title','question_body','answer']

In [5]:
lang = Path('../input/google-quest-language-model/')

In [6]:
BS =128

In [7]:
data_lm = load_data(lang, 'data_lm_export.pkl', bs=BS )

NameError: name 'load_data' is not defined

In [8]:
from scipy.stats import spearmanr

class AvgSpearman(Callback):
    def on_epoch_begin(self, **kwargs):
        self.preds = None
        self.target = None
    
    def on_batch_end(self, last_output, last_target, **kwargs):
        if self.preds is None or self.target is None:
            self.preds = last_output.cpu()
            self.target = last_target.cpu()
        else:
            self.preds = np.append(self.preds, last_output.cpu(), axis=0)
            self.target = np.append(self.target, last_target.cpu(), axis=0)
    
    def on_epoch_end(self, last_metrics, **kwargs):
        spearsum = 0
        for col in range(self.preds.shape[1]):
            spearsum += spearmanr(self.preds[:,col], self.target[:,col]).correlation
        res = spearsum / (self.preds.shape[1] + 1)
        return add_metrics(last_metrics, res)

NameError: name 'Callback' is not defined

In [9]:
colQA = ['question_asker_intent_understanding',
       'question_body_critical', 'question_conversational',
       'question_expect_short_answer', 'question_fact_seeking',
       'question_has_commonly_accepted_answer',
       'question_interestingness_others', 'question_interestingness_self',
       'question_multi_intent', 'question_not_really_a_question',
       'question_opinion_seeking', 'question_type_choice',
       'question_type_compare', 'question_type_consequence',
       'question_type_definition', 'question_type_entity',
       'question_type_instructions', 'question_type_procedure',
       'question_type_reason_explanation', 'question_type_spelling',
       'question_well_written', 'answer_helpful',
       'answer_level_of_information', 'answer_plausible', 'answer_relevance',
       'answer_satisfaction', 'answer_type_instructions',
       'answer_type_procedure', 'answer_type_reason_explanation',
       'answer_well_written']

ques = colQA[:-9]
ans = colQA[-9:]

In [10]:
ts_q = (TextList.from_df(test, path, vocab=data_lm.vocab, cols=cols[:2]))
ts_a = (TextList.from_df(test, path, vocab=data_lm.vocab, cols=cols[-1]))

NameError: name 'TextList' is not defined

In [11]:
data_q = (TextList.from_df(train, path, vocab=data_lm.vocab, cols=cols[:2])
    .split_by_rand_pct(0.2, seed=42)
    .label_from_df(cols=ques)
    .add_test(ts_q)
    .databunch(bs=128, num_workers=1))

NameError: name 'TextList' is not defined

In [12]:
data_a = (TextList.from_df(train, path, vocab=data_lm.vocab, cols=cols[-1])
    .split_by_rand_pct(0.2, seed=42)
    .label_from_df(cols=ans)
    .add_test(ts_a)
    .databunch(bs=128, num_workers=1))

NameError: name 'TextList' is not defined

In [13]:
q_learn = text_classifier_learner(data_q, AWD_LSTM,pretrained=False,
                                metrics=[AvgSpearman()],
                                 model_dir = '/kaggle/working/').to_fp16()
q_learn.load_encoder("../input/google-quest-language-model/ft_enc");

NameError: name 'text_classifier_learner' is not defined

In [14]:
a_learn = text_classifier_learner(data_a, AWD_LSTM,pretrained=False,
                                metrics=[AvgSpearman()],
                                model_dir = '/kaggle/working/').to_fp16()
a_learn.load_encoder("../input/google-quest-language-model/ft_enc");

NameError: name 'text_classifier_learner' is not defined

In [15]:
q_learn.load('../input/fastai-google-quest-q-and-a-classifier/best_model_ques');
a_learn.load('../input/fastai-google-quest-q-and-a-classifier/best_model_ans');

NameError: name 'q_learn' is not defined

In [16]:
a_preds, _ = a_learn.get_preds(DatasetType.Test, ordered=True)
q_preds, _ = q_learn.get_preds(DatasetType.Test, ordered=True)

NameError: name 'a_learn' is not defined

In [17]:
sample = pd.read_csv('../input/google-quest-challenge/sample_submission.csv')

NameError: name 'pd' is not defined

In [18]:
sample.loc[:, ques] = q_preds.numpy()
sample.loc[:, ans] = a_preds.numpy()
sample.to_csv("submission.csv", index=False)

NameError: name 'q_preds' is not defined

In [19]:
sample.head(10)

NameError: name 'sample' is not defined